In [ ]:
### save MFCCs in zips with 1000 audio patchs

import os, librosa, numpy as np, pandas as pd, shutil
from tqdm.auto import tqdm

# --- 1. Config ---
MASTER_CSV = '/content/drive/MyDrive/BUU_PHD_THESIS/master_mos_dataset.csv'
WAV_DIR = '/content/wav_files' # Local WAVs
GDRIVE_OUT = '/content/drive/MyDrive/BUU_PHD_THESIS/MFCC_PATCHES/'
PATCH_SIZE = 1000

os.makedirs(GDRIVE_OUT, exist_ok=True)
df = pd.read_csv(MASTER_CSV)

# --- 2. Parameters ---
SR, N_MFCC, N_FFT, HOP_LENGTH = 16000, 40, 512, 256
SEG_DUR = 5
SAMPLES_PER_SEG = SR * SEG_DUR

def process_batch(batch_df, patch_idx):
    local_patch_dir = f'/content/patch_{patch_idx:03d}'
    os.makedirs(local_patch_dir, exist_ok=True)

    for _, row in batch_df.iterrows():
        file_name = row[0]
        audio_id = os.path.splitext(file_name)[0]
        file_path = os.path.join(WAV_DIR, file_name)

        if not os.path.exists(file_path): continue

        try:
            y, _ = librosa.load(file_path, sr=SR)
            file_out = os.path.join(local_patch_dir, audio_id)
            os.makedirs(file_out, exist_ok=True)

            num_segs = int(np.ceil(len(y) / SAMPLES_PER_SEG))
            meta = []
            for i in range(num_segs):
                y_seg = y[i*SAMPLES_PER_SEG : (i+1)*SAMPLES_PER_SEG]
                valid_f = 1 + (len(y_seg) // HOP_LENGTH)
                if len(y_seg) < SAMPLES_PER_SEG:
                    y_seg = np.pad(y_seg, (0, SAMPLES_PER_SEG - len(y_seg)), 'constant')

                mfcc = librosa.feature.mfcc(y=y_seg, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
                seg_name = f"{audio_id}_seg_{i:04d}.npy"
                np.save(os.path.join(file_out, seg_name), mfcc)
                meta.append({'segment_path': seg_name, 'valid_frames': valid_f})

            pd.DataFrame(meta).to_csv(os.path.join(file_out, "segment_info.csv"), index=False)
        except: continue

    # Zip and Move to Drive
    zip_path = f'/content/patch_{patch_idx:03d}'
    shutil.make_archive(zip_path, 'zip', local_patch_dir)
    shutil.move(f"{zip_path}.zip", os.path.join(GDRIVE_OUT, f"patch_{patch_idx:03d}.zip"))
    shutil.rmtree(local_patch_dir) # Clear local space

# --- 3. Execute in Patches ---
total_patches = int(np.ceil(len(df) / PATCH_SIZE))
for p in tqdm(range(total_patches), desc="Total Progress"):
    start_idx = p * PATCH_SIZE
    end_idx = start_idx + PATCH_SIZE
    process_batch(df.iloc[start_idx:end_idx], p + 1)